# Umbral por Cluster sobre la Probabilidad del Transformer

Este notebook evalua una estrategia mas ajustada al caso `binary_paper`:

- tomamos la probabilidad del Transformer para la clase de riesgo
- ajustamos un umbral distinto por `cluster_id` usando `validation`
- aplicamos esos umbrales en `test`

Estrategia metodologica:

- calibracion del umbral con `validation`
- evaluacion final con `test`

Eso evita leakage respecto al Transformer base y permite medir si el clustering aporta valor como mecanismo de recalibracion.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

In [17]:
PROJECT_ROOT = Path('/workspace/TFM_education_ai_analytics')
TRANSFORMER_SCOPE = 'binary_classification_paper'
TRANSFORMER_DIR = PROJECT_ROOT / 'data' / '7_model_predictions' / TRANSFORMER_SCOPE
CLUSTERING_DIR = PROJECT_ROOT / 'data' / '5_students_segmented'

print('Transformer dir:', TRANSFORMER_DIR)
print('Clustering dir :', CLUSTERING_DIR)
print('Transformer exists:', TRANSFORMER_DIR.exists())
print('Clustering exists :', CLUSTERING_DIR.exists())

Transformer dir: /workspace/TFM_education_ai_analytics/data/7_model_predictions/binary_classification_paper
Clustering dir : /workspace/TFM_education_ai_analytics/data/5_students_segmented
Transformer exists: True
Clustering exists : True


In [18]:
WEEK_PATTERN = re.compile(r'W(\d+)')

def extract_week(path: Path) -> int:
    match = WEEK_PATTERN.search(path.stem)
    if match is None:
        raise ValueError(f'No se pudo extraer la semana desde: {path.name}')
    return int(match.group(1))

def available_weeks() -> list[int]:
    val_files = sorted((TRANSFORMER_DIR / 'validation').glob('transformer_preds_uptW*.csv'))
    test_files = sorted((TRANSFORMER_DIR / 'test').glob('transformer_preds_uptW*.csv'))
    cluster_val_files = sorted((CLUSTERING_DIR / 'validation').glob('students_segmented_uptoW*.csv'))
    cluster_test_files = sorted((CLUSTERING_DIR / 'test').glob('students_segmented_uptoW*.csv'))

    val_weeks = {extract_week(path) for path in val_files}
    test_weeks = {extract_week(path) for path in test_files}
    cluster_val_weeks = {extract_week(path) for path in cluster_val_files}
    cluster_test_weeks = {extract_week(path) for path in cluster_test_files}

    return sorted(val_weeks & test_weeks & cluster_val_weeks & cluster_test_weeks)

WEEKS = available_weeks()
WEEKS

[1, 3, 5, 8, 10, 12, 15, 18, 20, 24, 28]

In [19]:
def load_transformer(split: str, week: int) -> pd.DataFrame:
    path = TRANSFORMER_DIR / split / f'transformer_preds_uptW{week}.csv'
    df = pd.read_csv(path)
    return df.copy()

def load_clustering(split: str, week: int) -> pd.DataFrame:
    path = CLUSTERING_DIR / split / f'students_segmented_uptoW{week}.csv'
    df = pd.read_csv(path)
    df = df.rename(columns={'unique_id': 'id_student'})
    return df.copy()

def build_dataset(split: str, week: int) -> pd.DataFrame:
    trf = load_transformer(split=split, week=week)
    clu = load_clustering(split=split, week=week)
    merged = trf.merge(clu, on='id_student', how='inner', validate='one_to_one')
    merged['week'] = int(week)
    return merged

sample_val = build_dataset('validation', WEEKS[0])
sample_val.head()

,id_student,classification_scope,target_tag,prob_pass_dist,prob_withdrawn,pred_class_id,pred_class_name,true_class_id,true_class_name,cluster_id,...,cluster_name,p_cluster_0,p_cluster_1,p_cluster_2,p_cluster_3,p_cluster_4,confidence,entropy,entropy_norm,week
0,1023623_CCC_2014J,binary_classification_paper,2clases_paper,0.444875,0.555125,1,Withdrawn,0,Pass/Dist,2,...,Perfil intermedio B,0.0,5.873461e-07,1.000000,6.425426e-09,0.0,1.000000,9.025075e-06,5.607594e-06,1
1,1044992_DDD_2013B,binary_classification_paper,2clases_paper,0.534384,0.465616,0,Pass/Dist,0,Pass/Dist,2,...,Perfil intermedio B,0.0,2.576098e-07,1.000000,1.515543e-10,0.0,1.000000,4.209858e-06,2.615732e-06,1
2,105527_BBB_2013J,binary_classification_paper,2clases_paper,0.532778,0.467222,0,Pass/Dist,0,Pass/Dist,1,...,Alto rendimiento (estratégico),0.0,9.998681e-01,0.000132,6.496001e-12,0.0,0.999868,1.310227e-03,8.140901e-04,1
3,105939_FFF_2014J,binary_classification_paper,2clases_paper,0.502113,0.497887,0,Pass/Dist,0,Pass/Dist,2,...,Perfil intermedio B,0.0,1.463476e-10,1.000000,1.577955e-10,0.0,1.000000,6.873453e-09,4.270716e-09,1
4,1082062_DDD_2014J,binary_classification_paper,2clases_paper,0.536444,0.463556,0,Pass/Dist,0,Pass/Dist,2,...,Perfil intermedio B,0.0,9.870535e-03,0.990129,9.846163e-09,0.0,0.990129,5.540596e-02,3.442565e-02,1


In [20]:
sample_val.columns.tolist()

['id_student',
 'classification_scope',
 'target_tag',
 'prob_pass_dist',
 'prob_withdrawn',
 'pred_class_id',
 'pred_class_name',
 'true_class_id',
 'true_class_name',
 'cluster_id',
 'cluster_label',
 'cluster_name',
 'p_cluster_0',
 'p_cluster_1',
 'p_cluster_2',
 'p_cluster_3',
 'p_cluster_4',
 'confidence',
 'entropy',
 'entropy_norm',
 'week']

## Definicion de la estrategia

En lugar de entrenar un meta-modelo global, usamos:

- la probabilidad `prob_withdrawn` del Transformer
- el `cluster_id` asignado por el clustering
- un umbral distinto por cluster aprendido en `validation`

La hipotesis es simple: el Transformer puede ordenar bien a los estudiantes, pero el punto de corte optimo puede variar segun el segmento.

In [21]:
TARGET_COL = 'true_class_id'
PROB_COL = 'prob_withdrawn'
CLUSTER_COL = 'cluster_id'
DEFAULT_THRESHOLD = 0.5
THRESHOLD_GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)
MIN_CLUSTER_SAMPLES = 25

strategy_preview = sample_val[[
    'id_student',
    PROB_COL,
    CLUSTER_COL,
    'cluster_label',
    TARGET_COL,
    'pred_class_id',
]].copy()

print('Probability column:', PROB_COL)
print('Cluster column    :', CLUSTER_COL)
print('Default threshold :', DEFAULT_THRESHOLD)
print('Thresholds tested :', len(THRESHOLD_GRID))
strategy_preview.head()

Categorical: ['pred_class_id', 'cluster_id', 'cluster_label']
Numeric count: 11


['prob_pass_dist',
 'prob_withdrawn',
 'pred_class_id',
 'cluster_id',
 'cluster_label',
 'p_cluster_0',
 'p_cluster_1',
 'p_cluster_2',
 'p_cluster_3',
 'p_cluster_4',
 'confidence',
 'entropy',
 'entropy_norm',
 'week']

In [ ]:
def find_best_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> tuple[float, float]:
    best_threshold = DEFAULT_THRESHOLD
    best_score = float('-inf')

    for threshold in THRESHOLD_GRID:
        y_pred = (y_prob >= threshold).astype(int)
        score = balanced_accuracy_score(y_true, y_pred)
        if score > best_score + 1e-12 or (abs(score - best_score) <= 1e-12 and abs(threshold - 0.5) < abs(best_threshold - 0.5)):
            best_score = float(score)
            best_threshold = float(threshold)

    return best_threshold, best_score

def fit_cluster_thresholds(df_val: pd.DataFrame) -> pd.DataFrame:
    rows = []
    global_threshold, global_bacc = find_best_threshold(
        df_val[TARGET_COL].astype(int).to_numpy(),
        df_val[PROB_COL].to_numpy(),
    )

    for cluster_id, cluster_df in df_val.groupby(CLUSTER_COL, dropna=False):
        y_true = cluster_df[TARGET_COL].astype(int).to_numpy()
        y_prob = cluster_df[PROB_COL].to_numpy()

        if len(cluster_df) < MIN_CLUSTER_SAMPLES or len(np.unique(y_true)) < 2:
            best_threshold = global_threshold
            best_bacc = global_bacc
            source = 'global_fallback'
        else:
            best_threshold, best_bacc = find_best_threshold(y_true, y_prob)
            source = 'cluster_fit'

        rows.append({
            CLUSTER_COL: cluster_id,
            'n_val_cluster': int(len(cluster_df)),
            'threshold': float(best_threshold),
            'val_balanced_accuracy_cluster': float(best_bacc),
            'threshold_source': source,
        })

    thresholds_df = pd.DataFrame(rows).sort_values(CLUSTER_COL).reset_index(drop=True)
    thresholds_df.attrs['global_threshold'] = float(global_threshold)
    thresholds_df.attrs['global_balanced_accuracy'] = float(global_bacc)
    return thresholds_df

def apply_cluster_thresholds(df: pd.DataFrame, thresholds_df: pd.DataFrame) -> pd.DataFrame:
    global_threshold = thresholds_df.attrs['global_threshold']
    out = df.merge(
        thresholds_df[[CLUSTER_COL, 'threshold', 'threshold_source']],
        on=CLUSTER_COL,
        how='left',
    )
    out['threshold'] = out['threshold'].fillna(global_threshold)
    out['threshold_source'] = out['threshold_source'].fillna('missing_cluster_fallback')
    out['y_prob'] = out[PROB_COL].astype(float)
    out['y_pred'] = (out['y_prob'] >= out['threshold']).astype(int)
    return out

thresholds_preview = fit_cluster_thresholds(build_dataset('validation', WEEKS[0]))
display(thresholds_preview)
print('Global fallback threshold:', thresholds_preview.attrs['global_threshold'])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transforme

In [ ]:
def evaluate_window(week: int) -> dict:
    df_val = build_dataset('validation', week)
    df_test = build_dataset('test', week)

    thresholds_df = fit_cluster_thresholds(df_val)
    scored_test = apply_cluster_thresholds(df_test, thresholds_df)

    y_test = scored_test[TARGET_COL].astype(int).to_numpy()
    y_pred = scored_test['y_pred'].to_numpy()
    y_prob = scored_test['y_prob'].to_numpy()

    metrics = {
        'week': int(week),
        'n_val': int(len(df_val)),
        'n_test': int(len(df_test)),
        'global_threshold': float(thresholds_df.attrs['global_threshold']),
        'n_clusters': int(thresholds_df[CLUSTER_COL].nunique()),
        'n_cluster_fallbacks': int((thresholds_df['threshold_source'] != 'cluster_fit').sum()),
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test, y_pred)),
        'precision': float(precision_score(y_test, y_pred, zero_division=0)),
        'recall': float(recall_score(y_test, y_pred, zero_division=0)),
        'f1_score': float(f1_score(y_test, y_pred, zero_division=0)),
        'auc': float(roc_auc_score(y_test, y_prob)),
    }
    return metrics

results = [evaluate_window(week) for week in WEEKS]
results_df = pd.DataFrame(results).sort_values('week').reset_index(drop=True)
results_df

,week,n_val,n_test,accuracy,balanced_accuracy,precision,recall,f1_score,auc
0,1,2774,2858,0.684395,0.659760,0.445173,0.605096,0.512959,0.704945
1,3,3203,3268,0.750612,0.715320,0.564850,0.630640,0.595935,0.783310
2,5,3252,3304,0.788136,0.751954,0.633955,0.663926,0.648594,0.824409
3,8,3283,3321,0.839807,0.798410,0.744287,0.697248,0.720000,0.866518
4,10,3290,3330,0.852853,0.820822,0.754912,0.742625,0.748718,0.880687
5,12,3291,3331,0.856199,0.829987,0.751497,0.766022,0.758690,0.899585
6,15,3295,3336,0.894784,0.868428,0.833684,0.804061,0.818605,0.924226
7,18,3297,3337,0.913995,0.896209,0.855397,0.852792,0.854093,0.949919
8,20,3299,3338,0.936189,0.915783,0.913276,0.865990,0.889005,0.955408
9,24,3299,3339,0.939203,0.927403,0.895854,0.898580,0.897215,0.972834


## Contraste con el Transformer base

A continuacion recuperamos las tablas resumen del Transformer base para `binary_paper` y las comparamos directamente con la estrategia de umbral por cluster.

La idea es responder tres preguntas:

- en que semanas gana la recalibracion por cluster
- en que metricas gana de verdad
- si el clustering aporta valor como ajuste del punto de corte o no

In [24]:
baseline_val_path = PROJECT_ROOT / 'reports' / 'transformer_training' / 'binary_classification_paper' / 'validation_summary_2clases_paper.csv'
baseline_test_path = PROJECT_ROOT / 'reports' / 'transformer_training' / 'binary_classification_paper' / 'test_summary_2clases_paper.csv'

baseline_val_df = pd.read_csv(baseline_val_path)
baseline_test_df = pd.read_csv(baseline_test_path)

print('Baseline validation path:', baseline_val_path)
print('Baseline test path      :', baseline_test_path)
display(baseline_test_df.head())

Baseline validation path: /workspace/TFM_education_ai_analytics/reports/transformer_training/binary_classification_paper/validation_summary_2clases_paper.csv
Baseline test path      : /workspace/TFM_education_ai_analytics/reports/transformer_training/binary_classification_paper/test_summary_2clases_paper.csv


,upto_week,n_samples,accuracy,balanced_accuracy,precision,recall,auc
0,1,2858,0.673548,0.662968,0.435764,0.639490,0.703246
1,3,3268,0.757344,0.718528,0.577519,0.625393,0.783689
2,5,3304,0.791465,0.754015,0.641153,0.662898,0.826103
3,8,3321,0.828064,0.793037,0.709611,0.707441,0.863550
4,10,3330,0.852853,0.822596,0.751788,0.748728,0.883262


In [25]:
comparison_df = results_df.rename(columns={'week': 'upto_week'}).merge(
    baseline_test_df,
    on='upto_week',
    suffixes=('_ensemble', '_transformer'),
    how='inner',
)

metrics_to_compare = ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'auc']
for metric in metrics_to_compare:
    comparison_df[f'delta_{metric}'] = comparison_df[f'{metric}_ensemble'] - comparison_df[f'{metric}_transformer']

comparison_view = comparison_df[[
    'upto_week',
    'accuracy_ensemble', 'accuracy_transformer', 'delta_accuracy',
    'balanced_accuracy_ensemble', 'balanced_accuracy_transformer', 'delta_balanced_accuracy',
    'precision_ensemble', 'precision_transformer', 'delta_precision',
    'recall_ensemble', 'recall_transformer', 'delta_recall',
    'auc_ensemble', 'auc_transformer', 'delta_auc',
]].copy()

comparison_view = comparison_view.sort_values('upto_week').reset_index(drop=True)
comparison_view

,upto_week,accuracy_ensemble,accuracy_transformer,delta_accuracy,balanced_accuracy_ensemble,balanced_accuracy_transformer,delta_balanced_accuracy,precision_ensemble,precision_transformer,delta_precision,recall_ensemble,recall_transformer,delta_recall,auc_ensemble,auc_transformer,delta_auc
0,1,0.684395,0.673548,0.010847,0.659760,0.662968,-0.003208,0.445173,0.435764,0.009409,0.605096,0.639490,-0.034395,0.704945,0.703246,0.001699
1,3,0.750612,0.757344,-0.006732,0.715320,0.718528,-0.003208,0.564850,0.577519,-0.012670,0.630640,0.625393,0.005247,0.783310,0.783689,-0.000379
2,5,0.788136,0.791465,-0.003329,0.751954,0.754015,-0.002060,0.633955,0.641153,-0.007198,0.663926,0.662898,0.001028,0.824409,0.826103,-0.001694
3,8,0.839807,0.828064,0.011743,0.798410,0.793037,0.005373,0.744287,0.709611,0.034676,0.697248,0.707441,-0.010194,0.866518,0.863550,0.002969
4,10,0.852853,0.852853,0.000000,0.820822,0.822596,-0.001774,0.754912,0.751788,0.003125,0.742625,0.748728,-0.006104,0.880687,0.883262,-0.002574
5,12,0.856199,0.854398,0.001801,0.829987,0.831962,-0.001975,0.751497,0.741748,0.009749,0.766022,0.777213,-0.011190,0.899585,0.900359,-0.000774
6,15,0.894784,0.888789,0.005995,0.868428,0.866534,0.001894,0.833684,0.811359,0.022325,0.804061,0.812183,-0.008122,0.924226,0.925253,-0.001027
7,18,0.913995,0.899011,0.014984,0.896209,0.892955,0.003254,0.855397,0.799445,0.055952,0.852792,0.878173,-0.025381,0.949919,0.950309,-0.000390
8,20,0.936189,0.927202,0.008987,0.915783,0.914130,0.001653,0.913276,0.872490,0.040786,0.865990,0.882234,-0.016244,0.955408,0.957821,-0.002413
9,24,0.939203,0.938005,0.001198,0.927403,0.928615,-0.001212,0.895854,0.886792,0.009062,0.898580,0.905680,-0.007099,0.972834,0.974786,-0.001952


In [26]:
summary_rows = []
for metric in metrics_to_compare:
    delta_col = f'delta_{metric}'
    wins = int((comparison_df[delta_col] > 0).sum())
    losses = int((comparison_df[delta_col] < 0).sum())
    ties = int((comparison_df[delta_col].abs() < 1e-12).sum())
    summary_rows.append({
        'metric': metric,
        'mean_delta': comparison_df[delta_col].mean(),
        'median_delta': comparison_df[delta_col].median(),
        'max_gain': comparison_df[delta_col].max(),
        'max_loss': comparison_df[delta_col].min(),
        'weeks_won': wins,
        'weeks_lost': losses,
        'weeks_tied': ties,
    })

contrast_summary_df = pd.DataFrame(summary_rows).sort_values('mean_delta', ascending=False)
contrast_summary_df

,metric,mean_delta,median_delta,max_gain,max_loss,weeks_won,weeks_lost,weeks_tied
2,precision,0.015783,0.009409,0.055952,-0.012670,9,2,0
0,accuracy,0.004381,0.002695,0.014984,-0.006732,8,2,1
1,balanced_accuracy,0.000059,-0.001212,0.005373,-0.003208,5,6,0
4,auc,-0.000664,-0.000774,0.002969,-0.002574,2,9,0
3,recall,-0.010223,-0.008122,0.005247,-0.034395,2,8,1


In [ ]:
best_bacc_row = comparison_df.loc[comparison_df['delta_balanced_accuracy'].idxmax()]
worst_bacc_row = comparison_df.loc[comparison_df['delta_balanced_accuracy'].idxmin()]
mean_bacc_delta = comparison_df['delta_balanced_accuracy'].mean()
mean_auc_delta = comparison_df['delta_auc'].mean()

print('Lectura rapida del contraste umbral por cluster vs transformer base')
print(f"- Balanced accuracy media: {mean_bacc_delta:+.4f}")
print(f"- AUC media: {mean_auc_delta:+.4f}")
print(
    f"- Mejor semana en balanced accuracy para el umbral por cluster: W{int(best_bacc_row['upto_week']):02d} "
    f"({best_bacc_row['delta_balanced_accuracy']:+.4f})"
)
print(
    f"- Peor semana en balanced accuracy para el umbral por cluster: W{int(worst_bacc_row['upto_week']):02d} "
    f"({worst_bacc_row['delta_balanced_accuracy']:+.4f})"
)

if mean_bacc_delta > 0.01:
    verdict = 'La recalibracion por cluster mejora al transformer base de forma consistente y con una magnitud apreciable.'
elif mean_bacc_delta > 0:
    verdict = 'La recalibracion por cluster mejora ligeramente al transformer base, pero la ganancia media es pequena.'
elif mean_bacc_delta > -0.01:
    verdict = 'La recalibracion por cluster y el transformer base quedan muy parejos; las diferencias medias son marginales.'
else:
    verdict = 'La recalibracion por cluster empeora frente al transformer base con una caida media relevante.'

print(f'- Veredicto: {verdict}')

Lectura rápida del contraste ensemble vs transformer base
- Balanced accuracy media: +0.0001
- AUC media: -0.0007
- Mejor semana en balanced accuracy para el ensemble: W08 (+0.0054)
- Peor semana en balanced accuracy para el ensemble: W03 (-0.0032)
- Veredicto: El ensemble mejora ligeramente al transformer base, pero la ganancia media es pequeña.


In [28]:
summary = pd.DataFrame({
    'metric': ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1_score', 'auc'],
    'mean': [results_df[c].mean() for c in ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1_score', 'auc']],
    'max': [results_df[c].max() for c in ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1_score', 'auc']],
    'min': [results_df[c].min() for c in ['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1_score', 'auc']],
})
summary

,metric,mean,max,min
0,accuracy,0.855949,0.959269,0.684395
1,balanced_accuracy,0.830494,0.951362,0.659760
2,precision,0.756641,0.930162,0.445173
3,recall,0.769003,0.932049,0.605096
4,f1_score,0.761356,0.931104,0.512959
5,auc,0.885773,0.981656,0.704945


In [29]:
best_by_bacc = results_df.sort_values(['balanced_accuracy', 'f1_score', 'auc'], ascending=False).reset_index(drop=True)
best_by_bacc

,week,n_val,n_test,accuracy,balanced_accuracy,precision,recall,f1_score,auc
0,28,3299,3339,0.959269,0.951362,0.930162,0.932049,0.931104,0.981656
1,24,3299,3339,0.939203,0.927403,0.895854,0.898580,0.897215,0.972834
2,20,3299,3338,0.936189,0.915783,0.913276,0.865990,0.889005,0.955408
3,18,3297,3337,0.913995,0.896209,0.855397,0.852792,0.854093,0.949919
4,15,3295,3336,0.894784,0.868428,0.833684,0.804061,0.818605,0.924226
5,12,3291,3331,0.856199,0.829987,0.751497,0.766022,0.758690,0.899585
6,10,3290,3330,0.852853,0.820822,0.754912,0.742625,0.748718,0.880687
7,8,3283,3321,0.839807,0.798410,0.744287,0.697248,0.720000,0.866518
8,5,3252,3304,0.788136,0.751954,0.633955,0.663926,0.648594,0.824409
9,3,3203,3268,0.750612,0.715320,0.564850,0.630640,0.595935,0.783310


In [ ]:
WEEK_TO_INSPECT = int(best_by_bacc.loc[0, 'week'])
df_val = build_dataset('validation', WEEK_TO_INSPECT)
thresholds_df = fit_cluster_thresholds(df_val)

global_threshold = thresholds_df.attrs['global_threshold']
thresholds_inspect_df = thresholds_df.merge(
    df_val.groupby(CLUSTER_COL).agg(
        val_students=(TARGET_COL, 'size'),
        positive_rate=(TARGET_COL, 'mean'),
        mean_transformer_prob=(PROB_COL, 'mean'),
    ).reset_index(),
    on=CLUSTER_COL,
    how='left',
).sort_values(['threshold_source', 'n_val_cluster', CLUSTER_COL], ascending=[True, False, True])

print('Semana inspeccionada:', WEEK_TO_INSPECT)
print('Global threshold  :', round(global_threshold, 4))
display(thresholds_inspect_df)

Semana inspeccionada: 28


,feature,coef,abs_coef
1,num__prob_withdrawn,1.670011,1.670011
0,num__prob_pass_dist,-1.670010,1.670010
11,cat__pred_class_id_0,-0.501321,0.501321
17,cat__cluster_label_CONSISTENT_GOOD,0.445125,0.445125
15,cat__cluster_id_2,0.445125,0.445125
4,num__p_cluster_2,-0.413317,0.413317
7,num__confidence,0.374027,0.374027
2,num__p_cluster_0,0.364402,0.364402
14,cat__cluster_id_1,-0.314816,0.314816
18,cat__cluster_label_METHODICAL_EXPLORER,-0.314816,0.314816


## Interpretacion rapida

Si esta recalibracion mejora o iguala al Transformer en balanced accuracy o F1, hay senal de que el clustering aporta valor ajustando el punto de corte segun el segmento.

Si empeora o apenas cambia, lo mas probable es que:

- el Transformer ya este bien calibrado con un umbral global
- el clustering no se alinee con diferencias reales de decision
- haga falta una estrategia mas selectiva, por ejemplo activar la correccion solo en casos inciertos